# Nhánh 1 — So sánh kiến trúc

So sánh 4 candidate cho bộ phân loại SQLi đa lớp, chọn theo **F1-macro vs latency vs size** (không mặc định chọn transformer).

Dữ liệu: `data/processed/branch1_train.csv` (**67.796 dòng, 5 lớp**, train/test 54.236/13.560). Lớp `stacked` (chỉ 363 mẫu synthetic) đã bị loại khỏi schema, nên toàn bộ so sánh được **train lại trên 5 lớp**; CNN và DistilBERT có head đúng 5 lớp (số lớp suy ra từ dữ liệu, không hardcode).

Kết quả số liệu đọc từ `report/metrics/branch1_architecture_comparison.json` (sinh bởi `train/compare_branch1_architectures.py`).

In [1]:
import json
import pandas as pd

with open('../../report/metrics/branch1_architecture_comparison.json') as f:
    results = json.load(f)

rows = []
for name, m in results.items():
    rows.append({
        'model': name,
        'F1_macro': round(m['f1_macro'], 4),
        'p50_ms': round(m['latency']['p50_ms'], 3),
        'p95_ms': round(m['latency']['p95_ms'], 3),
        'train_s': round(m['train_time_s'], 1),
        'size_MB': round(m['model_size_bytes']/1024/1024, 2),
    })
df = pd.DataFrame(rows)
df

,model,F1_macro,p50_ms,p95_ms,train_s,size_MB
0,tfidf_logreg,0.9822,0.801,1.184,17.3,3.51
1,tfidf_lightgbm,0.9912,91.672,104.703,328.0,5.69
2,distilbert,0.9892,2.863,4.357,1574.0,256.11
3,cnn_sqltok,0.9838,0.343,0.521,10.4,0.11


## Per-class F1

In [2]:
classes = ['normal', 'union_based', 'error_based', 'boolean_blind', 'time_blind']
per_class = {name: {c: round(m['per_class'][c]['f1-score'], 3) for c in classes} for name, m in results.items()}
pd.DataFrame(per_class).T

,normal,union_based,error_based,boolean_blind,time_blind
tfidf_logreg,0.956,0.995,0.999,0.961,1.000
tfidf_lightgbm,0.978,1.000,1.000,0.978,1.000
distilbert,0.977,0.999,0.997,0.974,0.998
cnn_sqltok,0.975,0.998,1.000,0.963,0.984


## Nhận xét & Quyết định

Số liệu 5 lớp (test 13.560 dòng):

| Model | F1-macro | p50 latency | Size | Ghi chú |
|---|---|---|---|---|
| TF-IDF + LogReg | 0.9822 | 0.80ms | 3.5MB | Nhanh, đơn giản |
| TF-IDF + LightGBM | 0.9912 | 91.7ms | 5.7MB | F1 cao nhất **nhưng latency ~92ms** (chậm gấp ~115x) |
| DistilBERT | 0.9892 | 2.86ms | 256MB | F1 cao, latency ổn (GPU), **size lớn + train ~26 phút** |
| CNN + SQL-tokenizer | 0.9838 | 0.34ms | 0.11MB | **Nhanh nhất, nhỏ nhất (28.485 params)**, F1 sát nhóm dẫn đầu |

**Quyết định: chọn TF-IDF + LogReg** làm baseline production cho Nhánh 1.
- Chênh lệch F1 giữa 4 model rất nhỏ (0.982–0.991) → không đáng đánh đổi.
- LightGBM tuy F1 cao nhất nhưng latency ~92ms là quá cao cho database proxy real-time.
- DistilBERT không cho lợi ích F1 rõ rệt mà tốn 256MB + cần GPU + train lâu.
- CNN là ứng viên thay thế tốt (nhanh/nhỏ nhất) nếu cần thêm khả năng học đặc trưng.

> Weights của CNN + DistilBERT (5 lớp) đã đẩy lên HF: `Jason-42195/VNU-SQLi-Detection-Models`, thư mục `branch1_comparison/`.

⚠️ **Cảnh báo quan trọng:** F1 cao đồng loạt (~0.98–0.99) trên cả 4 kiến trúc → dấu hiệu dữ liệu **quá dễ phân biệt**, không phản ánh độ khó thật của tấn công che giấu (obfuscated). Con số F1 này **không nên hiểu là hệ thống đã "gần hoàn hảo"** — tập test adversarial mới là thước đo thật.